# Explorar la API del Seller Center de Falabella
Objetivo: confirmar que la conexión funciona, ver qué campos llegan y definir la limpieza.
**Antes:** copia `.env.example` a `.env` y llénalo con el usuario y la API Key.

In [ ]:
import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
from dotenv import load_dotenv
from falabella_client import FalabellaClient, normalizar_item

load_dotenv()
cli = FalabellaClient(os.environ["FALABELLA_USER_ID"], os.environ["FALABELLA_API_KEY"])

## 1. Traer órdenes de los últimos 7 días

In [ ]:
ordenes = cli.get_orders(created_after=datetime.now(timezone.utc) - timedelta(days=7))
print(len(ordenes), "órdenes")

## 2. Ver qué campos llegan
Solo los **nombres** de los campos (las órdenes traen datos personales del cliente: no los compartas).

In [ ]:
print(list(ordenes[0].keys()) if ordenes else "Sin órdenes en el rango")

## 3. Traer los ítems de cada orden y limpiar

In [ ]:
filas = []
for o in ordenes:
    for it in cli.get_order_items(o["OrderId"]):
        filas.append(normalizar_item(it, o))
df = pd.DataFrame(filas)
df["created_at"] = pd.to_datetime(df["created_at"])
df.head()

In [ ]:
# Estados que llegan: define cuáles cuentan como venta
df["estado"].value_counts()

## 4. Resumen diario (lo que alimentará el dashboard)

In [ ]:
ventas = df[~df["estado"].isin(["canceled", "cancelled", "failed", "returned"])]
diario = ventas.groupby(ventas["created_at"].dt.date).agg(
    ventas=("precio_pagado", "sum"), ordenes=("order_id", "nunique"))
diario

## 5. Revisar
- ¿Las horas de `created_at` coinciden con las del Seller Center (zona horaria)?
- ¿`precio_pagado` incluye envío o descuentos?
- ¿Qué estados hay que excluir? (ajustar `ESTADOS_EXCLUIDOS` en `.env`)

## 6. Probar el stock de productos (GetProducts)
Para calcular Sell-through y WOS. Revisa: ¿el nombre del producto coincide con el de ventas? ¿el SKU es el mismo formato?

In [ ]:
from falabella_client import normalizar_producto

productos = cli.get_products()
print(len(productos), "productos")
if productos:
    print(list(productos[0].keys()))

In [ ]:
prod_df = pd.DataFrame([normalizar_producto(p) for p in productos])
prod_df